# Approche Semi-supervisée

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from constants import FEATURES_DIR
import os
import mlflow
import mlflow.pytorch

# ======================
# Config
# ======================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
SEED = 42

# MLflow Config
MLFLOW_EXPERIMENT_NAME = "resnet50-weaklabels"

torch.manual_seed(SEED)
np.random.seed(SEED)

# ======================
# Dataset
# ======================
class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ======================
# ResNet50 head (classification) on top of ResNet50 embeddings (avgpool)
# ======================
class ResNet50Head(nn.Module):
    """
    Classification head qui prend en entrée des features extraites par ResNet50 (2048 dims via avgpool).
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)

# ======================
# Load data
# ======================
print("Chargement des features...")

files = [f for f in os.listdir(FEATURES_DIR) if f.endswith(".npy")]
X = np.load(FEATURES_DIR / "features_unlabeled_avgpool.npy")
y = np.load(FEATURES_DIR / "weak_labels.npy")

print(f"\n Samples weak-labeled: {len(y)}")
print(f"X shape: {X.shape}")
print(f"Distribution: {np.bincount(y)}")

# ======================
# Train / Val split
# ======================
dataset = FeatureDataset(X, y)

train_size = int(0.8 * len(dataset)) # 80% des données pour l'entrainement
val_size = len(dataset) - train_size # 20% des données pour la validation

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True) # shuffle : on s'assure qu'il n'y ait pas un ordre "caché" que le modèle mémoriserait
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ======================
# Training setup
# ======================
model = ResNet50Head(input_dim=X.shape[1]).to(DEVICE) # shape[1] sur tableau numpy = longueur du vecteur (2048), pré construit avec ResNet50
criterion = nn.CrossEntropyLoss() # mesure à quel point le modèle se trompe
optimizer = torch.optim.Adam(model.parameters(), lr=LR) # ajuste les poids du modèle pour réduire l'erreur

# ======================
# MLflow Run
# ======================
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run():
    # Log des hyperparamètres
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED,
        "device": DEVICE
    })

    print(f"\n Training on {DEVICE}...\n")

    for epoch in range(EPOCHS):
        model.train()
        train_loss_sum = 0.0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            
            train_loss_sum += loss.item()

        # Validation
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad(): # no_grad() = inférence seule, pas d'entrainement
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                preds = model(xb).argmax(1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(yb.numpy())

        # Calcul des métriques
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        avg_train_loss = train_loss_sum / len(train_loader)

        # Sauvegarde MLflow par epoch
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("val_acc", acc, step=epoch)
        mlflow.log_metric("val_f1", f1, step=epoch)
        mlflow.log_metric("val_precision", prec, step=epoch)
        mlflow.log_metric("val_recall", rec, step=epoch)

        print(f"Epoch {epoch+1:02d} | Loss={avg_train_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f} | Prec={prec:.4f} | Rec={rec:.4f}")

    # ======================
    # Final report (weak val)
    # ======================
    print("\n" + "=" * 50)
    print("Classification report (weak labels) [val split]:")
    print("=" * 50)
    report_weak = classification_report(y_true, y_pred, target_names=["Normal", "Cancer"], digits=4)
    print(report_weak)
    mlflow.log_text(report_weak, "reports/weak_labels_report.txt")

    # ======================
    # TEST DE VÉRITÉ : Évaluation sur les Strong Labels
    # ======================
    print("\n" + "=" * 50)
    print("TEST DE VÉRITÉ : Évaluation sur les STRONG LABELS")
    print("=" * 50)

    X_cancer_strong = np.load(FEATURES_DIR / "features_cancer_avgpool.npy")
    X_normal_strong = np.load(FEATURES_DIR / "features_normal_avgpool.npy")

    X_strong = np.vstack([X_cancer_strong, X_normal_strong])
    y_strong = np.array([1] * len(X_cancer_strong) + [0] * len(X_normal_strong))

    model.eval()
    X_strong_tensor = torch.tensor(X_strong, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        outputs = model(X_strong_tensor)
        y_pred_strong = outputs.argmax(1).cpu().numpy()

    # Métriques Strong
    acc_s = accuracy_score(y_strong, y_pred_strong)
    f1_s = f1_score(y_strong, y_pred_strong)
    prec_s = precision_score(y_strong, y_pred_strong, zero_division=0)
    rec_s = recall_score(y_strong, y_pred_strong, zero_division=0)

    mlflow.log_metrics({
        "strong_acc": acc_s,
        "strong_f1": f1_s,
        "strong_precision": prec_s,
        "strong_recall": rec_s
    })

    report_strong = classification_report(y_strong, y_pred_strong, target_names=["Normal", "Cancer"], digits=4)
    print(report_strong)
    mlflow.log_text(report_strong, "reports/strong_labels_report.txt")

    # ======================
    # Save
    # ======================
    save_path = FEATURES_DIR / "resnet50_head_weak_trained.pth"
    torch.save(model.state_dict(), save_path)
    print(f"\n Modèle sauvegardé: {save_path}")
    
    # Log du modèle dans MLflow
    mlflow.pytorch.log_model(model, "model")
    mlflow.log_artifact(str(save_path), "weights")

Chargement des features...

 Samples weak-labeled: 1406
X shape: (1406, 2048)
Distribution: [ 354 1052]


2026/01/13 09:22:52 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet




 Training on cpu...

Epoch 01 | Loss=0.4582 | Acc=0.9752 | F1=0.9837 | Prec=0.9860 | Rec=0.9814
Epoch 02 | Loss=0.1484 | Acc=0.9823 | F1=0.9884 | Prec=0.9861 | Rec=0.9907
Epoch 03 | Loss=0.0673 | Acc=0.9681 | F1=0.9794 | Prec=0.9640 | Rec=0.9953
Epoch 04 | Loss=0.0620 | Acc=0.9894 | F1=0.9930 | Prec=0.9953 | Rec=0.9907
Epoch 05 | Loss=0.0388 | Acc=0.9858 | F1=0.9907 | Prec=0.9907 | Rec=0.9907
Epoch 06 | Loss=0.0357 | Acc=0.9787 | F1=0.9858 | Prec=1.0000 | Rec=0.9721
Epoch 07 | Loss=0.0388 | Acc=0.9858 | F1=0.9907 | Prec=0.9862 | Rec=0.9953
Epoch 08 | Loss=0.0450 | Acc=0.9752 | F1=0.9839 | Prec=0.9727 | Rec=0.9953
Epoch 09 | Loss=0.0342 | Acc=0.9894 | F1=0.9930 | Prec=1.0000 | Rec=0.9860
Epoch 10 | Loss=0.0275 | Acc=0.9858 | F1=0.9907 | Prec=0.9862 | Rec=0.9953
Epoch 11 | Loss=0.0335 | Acc=0.9894 | F1=0.9930 | Prec=0.9907 | Rec=0.9953
Epoch 12 | Loss=0.0316 | Acc=0.9929 | F1=0.9953 | Prec=1.0000 | Rec=0.9907
Epoch 13 | Loss=0.0185 | Acc=0.9858 | F1=0.9907 | Prec=0.9907 | Rec=0.9907
Epo

2026/01/13 09:22:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 20 | Loss=0.0183 | Acc=0.9716 | F1=0.9816 | Prec=0.9726 | Rec=0.9907

Classification report (weak labels) [val split]:
              precision    recall  f1-score   support

      Normal     0.9683    0.9104    0.9385        67
      Cancer     0.9726    0.9907    0.9816       215

    accuracy                         0.9716       282
   macro avg     0.9704    0.9506    0.9600       282
weighted avg     0.9716    0.9716    0.9713       282


TEST DE VÉRITÉ : Évaluation sur les STRONG LABELS
              precision    recall  f1-score   support

      Normal     0.9778    0.8800    0.9263        50
      Cancer     0.8909    0.9800    0.9333        50

    accuracy                         0.9300       100
   macro avg     0.9343    0.9300    0.9298       100
weighted avg     0.9343    0.9300    0.9298       100


 Modèle sauvegardé: ../.data/features/resnet50_head_weak_trained.pth
🏃 View run smiling-gull-663 at: http://mlflow:5000/#/experiments/136364928939801491/runs/7ad58b39ab4

# Approche supervisée

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import mlflow
import mlflow.pytorch

# ======================
# BASELINE : Avec Train/Test Split
# ======================
print("\n" + "="*50)
print("VRAIE BASELINE : SUPERVISÉ AVEC SPLIT")
print("="*50)

# ======================
# MLflow Config (Supervised)
# ======================
MLFLOW_EXPERIMENT_NAME_SUP = "resnet50-supervised"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME_SUP)

with mlflow.start_run():
    # 1. Séparer les Strong Labels en Train (80%) et Test (20%)
    X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
        X_strong, y_strong, test_size=0.2, random_state=SEED, stratify=y_strong
    )

    # Log des hyperparamètres + infos split
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED,
        "device": DEVICE,
        "split": "train_test_split",
        "train_ratio": 0.8,
        "test_ratio": 0.2,
        "stratify": True,
        "input_dim": int(X.shape[1]),
        "n_train_strong": int(len(y_s_train)),
        "n_test_strong": int(len(y_s_test)),
    })

    # 2. Créer les loaders pour la baseline
    train_loader_s = DataLoader(
        FeatureDataset(X_s_train, y_s_train),
        batch_size=BATCH_SIZE,
        shuffle=True
    )  # shuffle : on s'assure qu'il n'y ait pas un ordre "caché" que le modèle mémoriserait

    test_tensor_s = torch.tensor(X_s_test, dtype=torch.float32).to(DEVICE)

    # 3. Nouveau modèle
    model_baseline = ResNet50Head(input_dim=X.shape[1]).to(DEVICE)  # shape[1] sur tableau numpy = longueur du vecteur (2048), pré construit avec ResNet50
    optimizer_b = torch.optim.Adam(model_baseline.parameters(), lr=LR)  # ajuste les poids du modèle pour réduire l'erreur

    # 4. Entraînement uniquement sur le TRAIN Strong
    for epoch in range(EPOCHS):
        model_baseline.train()
        train_loss_sum = 0.0
        train_n_batches = 0

        for xb, yb in train_loader_s:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            optimizer_b.zero_grad()
            preds = model_baseline(xb)
            loss = criterion(preds, yb)  # mesure à quel point le modèle se trompe
            loss.backward()
            optimizer_b.step()

            train_loss_sum += loss.item()
            train_n_batches += 1

        avg_train_loss = train_loss_sum / max(train_n_batches, 1)

        # (optionnel mais utile) : évaluer sur le TEST à chaque epoch pour suivre la généralisation
        model_baseline.eval()
        with torch.no_grad():  # no_grad() = inférence seule, pas d'entrainement
            y_pred_epoch = model_baseline(test_tensor_s).argmax(1).cpu().numpy()

        acc = accuracy_score(y_s_test, y_pred_epoch)
        f1 = f1_score(y_s_test, y_pred_epoch)
        prec = precision_score(y_s_test, y_pred_epoch, zero_division=0)
        rec = recall_score(y_s_test, y_pred_epoch, zero_division=0)

        # Log MLflow par epoch
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("test_acc", acc, step=epoch)
        mlflow.log_metric("test_f1", f1, step=epoch)
        mlflow.log_metric("test_precision", prec, step=epoch)
        mlflow.log_metric("test_recall", rec, step=epoch)

        print(f"Epoch {epoch+1:02d} | Loss={avg_train_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f} | Prec={prec:.4f} | Rec={rec:.4f}")

    # 5. Évaluation sur le TEST Strong (données jamais vues)
    model_baseline.eval()
    with torch.no_grad():
        y_pred_baseline = model_baseline(test_tensor_s).argmax(1).cpu().numpy()

    print("Rapport Baseline sur données INCONNUES :")
    report_sup = classification_report(
        y_s_test, y_pred_baseline,
        target_names=['Normal', 'Cancer'],
        digits=4
    )
    print(report_sup)

    # Log artifact : report texte
    mlflow.log_text(report_sup, "reports/supervised_strong_test_report.txt")

    # Log métriques finales (en plus de celles par epoch)
    mlflow.log_metrics({
        "final_test_acc": accuracy_score(y_s_test, y_pred_baseline),
        "final_test_f1": f1_score(y_s_test, y_pred_baseline),
        "final_test_precision": precision_score(y_s_test, y_pred_baseline, zero_division=0),
        "final_test_recall": recall_score(y_s_test, y_pred_baseline, zero_division=0),
    })

    # (optionnel) sauvegarde + log du modèle
    save_path_sup = FEATURES_DIR / "resnet50_head_supervised_trained.pth"
    torch.save(model_baseline.state_dict(), save_path_sup)
    print(f"\n Modèle supervisé sauvegardé: {save_path_sup}")

    mlflow.log_artifact(str(save_path_sup), "weights")
    mlflow.pytorch.log_model(model_baseline, "model")


VRAIE BASELINE : SUPERVISÉ AVEC SPLIT
Epoch 01 | Loss=0.6687 | Acc=0.5000 | F1=0.6667 | Prec=0.5000 | Rec=1.0000
Epoch 02 | Loss=0.5951 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 03 | Loss=0.5064 | Acc=0.9000 | F1=0.9091 | Prec=0.8333 | Rec=1.0000
Epoch 04 | Loss=0.3547 | Acc=0.9000 | F1=0.9091 | Prec=0.8333 | Rec=1.0000
Epoch 05 | Loss=0.3268 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 06 | Loss=0.2913 | Acc=0.9000 | F1=0.9000 | Prec=0.9000 | Rec=0.9000
Epoch 07 | Loss=0.2430 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 08 | Loss=0.1041 | Acc=0.9000 | F1=0.9091 | Prec=0.8333 | Rec=1.0000
Epoch 09 | Loss=0.1417 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 10 | Loss=0.0909 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 11 | Loss=0.0771 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 12 | Loss=0.1447 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Epoch 13 | Loss=0.0751 | Acc=0.9500 | F1=0.9524 | Prec=0.9091

2026/01/13 09:23:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 20 | Loss=0.0272 | Acc=0.9500 | F1=0.9524 | Prec=0.9091 | Rec=1.0000
Rapport Baseline sur données INCONNUES :
              precision    recall  f1-score   support

      Normal     1.0000    0.9000    0.9474        10
      Cancer     0.9091    1.0000    0.9524        10

    accuracy                         0.9500        20
   macro avg     0.9545    0.9500    0.9499        20
weighted avg     0.9545    0.9500    0.9499        20


 Modèle supervisé sauvegardé: ../.data/features/resnet50_head_supervised_trained.pth
🏃 View run industrious-vole-804 at: http://mlflow:5000/#/experiments/881879557287879727/runs/b8c7cdd213244a01ade7af3837beb458
🧪 View experiment at: http://mlflow:5000/#/experiments/881879557287879727
